In [1]:
import os
from glob import glob
from pathlib import Path
import yaml
import numpy as np
import pandas as pd
import geopandas as gpd

import matplotlib.pyplot as plt
import seaborn as sns

from python import prepare_inputs as pi
from python import solar_utils as su
from python import wind_utils as wu
from python import hydro_utils as hu
from python import climate_utils as cu
from python import utils as pu

from python.utils import project_path
from python.building_elec_model import building_types_dict

In [2]:
# Read zone shapefile for plots
nyiso_gdf = gpd.read_file(f"{project_path}/data/nyiso/gis/NYISO_Load_Zone_Dissolved.shp")

In [3]:
# Read run parameters
with open('./config.yml', 'r') as file:
    config = yaml.safe_load(file)

genX_file_name = config['genX_file_name']
genX_downscaled_file_name = config['genX_downscaled_file_name']
genX_iter = config['genX_iter']
busprop_name = config['busprop_name']
genprop_name = config['genprop_name']
climate_scenario_years = config['climate_scenario_years']
sim_years = config['sim_years']
resstock_upgrade = config['resstock_upgrade']
comstock_upgrade = config['comstock_upgrade']
run_name = config['run_name']

In [4]:
genX_file_name

'Combined_Capacity_S2_Low_RE_Mod_Elec'

# General

In [5]:
# Read genX
df_genX = pd.read_excel(f'{project_path}/data/genX/{genX_file_name}.xlsx')
df_genX = pi.tidy_genX(df_genX[df_genX['Iter'] == genX_iter]).query('EndCap > 1.')

In [6]:
df_genX.groupby('Resource')['EndCap'].sum()

Resource
battery                    5999.893968
biomass                     113.100000
distributed_generation     2553.998821
hydroelectric              4357.000000
hydroelectric_storage      1406.700000
natural_gas               16458.799716
nuclear                    2699.998092
offshore_wind              5737.997987
onshore_wind_existing      3388.999981
solar_existing             1706.399978
solar_new                 18648.858243
transportation              619.000000
water_heating               135.100000
Name: EndCap, dtype: float64

In [7]:
# Read GenX downscaled
df_genX_ds = pd.read_csv(f'{project_path}/data/genX/{genX_downscaled_file_name}.csv')

df_genX_ds = (df_genX_ds[(df_genX_ds['state'] == 'New York') & (df_genX_ds['iter'] == genX_iter)]
              .drop_duplicates(subset=['Resource', 'Zone'])
              .query('EndCap > 1.')
             )

In [8]:
df_genX_ds.groupby('technology')['EndCap'].sum()

technology
UtilityPV_Class1_Moderate_    18648.858243
Name: EndCap, dtype: float64

In [9]:
# Run name (make sure matches above!)
run_path = f"{project_path}/runs/{run_name}"

Path(f"{run_path}/inputs").mkdir(parents=True, exist_ok=True)
Path(f"{run_path}/outputs").mkdir(parents=True, exist_ok=True)
Path(f"{run_path}/figs").mkdir(parents=True, exist_ok=True)
Path(f"{run_path}/logs").mkdir(parents=True, exist_ok=True)

In [10]:
# Read ACORN generators
df_genprop_bo = pd.read_csv(f'{project_path}/data/grid/gen_prop_{busprop_name}.csv')
df_gencost_bo = pd.read_csv(f'{project_path}/data/grid/gencost_prop_{genprop_name}.csv')

# Merge costs and info
df_genprop = pd.merge(df_gencost_bo, df_genprop_bo)

# Select NYS only
df_genprop = df_genprop[df_genprop['GEN_ZONE'].isin(pu.zone_names)]

In [11]:
# Get climate input files
climate_paths = cu.generate_tgw_filelist(climate_scenario_years = climate_scenario_years,
                                         years = sim_years)

# Load

In [12]:
# Read baseline
df_baseline = pd.read_parquet(f"{project_path}/data/load/baseline/simulated/load_model_tgw_multizone_NN_{climate_scenario_years}.parquet")
df_baseline = df_baseline.groupby(['bus_id', 'datetime'])[['load_MW']].sum(numeric_only=True)

In [13]:
# Read ResStock
df_resstock = pd.concat([pd.read_parquet(f"{project_path}/data/load/resstock/simulated/bus_level/{climate_scenario_years}_{resstock_upgrade}_{home_type}.parquet") for home_type in building_types_dict['resstock']])
df_resstock = df_resstock.reset_index()
df_resstock = df_resstock.groupby(['bus_id','time']).sum()
df_resstock.index.names = ['bus_id', 'datetime']

In [14]:
# Read ComStock
df_comstock = pd.concat([pd.read_parquet(f"{project_path}/data/load/comstock/simulated/bus_level/{climate_scenario_years}_{comstock_upgrade}_{home_type}.parquet") for home_type in building_types_dict['comstock']])
df_comstock = df_comstock.reset_index()
df_comstock = df_comstock.groupby(['bus_id','time']).sum()
df_comstock.index.names = ['bus_id', 'datetime']

In [15]:
# Scale factors
genX_max_load = 27_055
resstock_scale_factor = float(0.5 * genX_max_load / df_resstock.groupby('datetime').sum().max())

TypeError: cannot convert the series to <class 'float'>

In [ ]:
# Combine
df_load = pd.concat([df_comstock.rename(columns={'bus_load_MW':'load_MW'}) * resstock_scale_factor,
                     df_resstock.rename(columns={'bus_load_MW':'load_MW'}) * resstock_scale_factor,
                     df_baseline]).groupby(['bus_id', 'datetime']).sum()

In [ ]:
# Plot
df_load_hourly = df_load.groupby('datetime').sum().reset_index().copy()
df_load_hourly['hourofyear'] = df_load_hourly['datetime'].dt.hour + (df_load_hourly['datetime'].dt.dayofyear - 1) * 24
df_load_hourly['load_GW'] = df_load_hourly['load_MW']/1000

fig, ax = plt.subplots(figsize=(10,4))

ax.axhline(y=27, color='C1', label='GenX maximum')
ax.legend()

df_load_hourly.groupby('hourofyear')['load_GW'].min().plot(color='silver', ax=ax, alpha=0.8)
df_load_hourly.groupby('hourofyear')['load_GW'].max().plot(color='silver', ax=ax, alpha=0.8)
df_load_hourly.groupby('hourofyear')['load_GW'].mean().plot(color='dimgray', ax=ax, alpha=0.8)

ax.set_xlabel('Hour of year')
ax.set_ylabel('NYSIO load [GW]')
ax.grid(0.2)

plt.savefig(f"{run_path}/figs/load_timeseries.pdf", bbox_inches='tight')

In [ ]:
# Store
df_load.reset_index().pivot(index='bus_id', 
                            columns='datetime', 
                            values='load_MW').to_csv(f"{run_path}/inputs/load_{climate_scenario_years}.csv")

# Non-renewable resources

In [36]:
PROJECT_DIR = "/home/fs01/jl2966/acorn-julia2/acorn-julia"
RUN_NAME    = "low_RE_mod_elec_iter0"    # e.g., "low_RE_mod_elec_iter0"
# --------------------------------

import os, time, numpy as np, pandas as pd

RUN_DIR = f"{PROJECT_DIR}/runs/{RUN_NAME}"
INPUTS  = f"{RUN_DIR}/inputs"
SA_PATH = f"{INPUTS}/storage_assignment.csv"

# Optionally, you can hardcode a bus_prop path here. If None, we'll auto-try two common files.
BUS_PROP_PATH = None  # e.g., "/.../data/grid/bus_prop_boyuan.csv"

print("=== PATHS IN USE ===")
print("PROJECT_DIR:", PROJECT_DIR)
print("RUN_NAME   :", RUN_NAME)
print("RUN_DIR    :", RUN_DIR)
print("INPUTS     :", INPUTS)
print("SA_PATH    :", SA_PATH)

if not os.path.exists(SA_PATH):
    raise FileNotFoundError(f"storage_assignment.csv not found at:\n  {SA_PATH}")

# 1) Load existing storage totals
df_base = pd.read_csv(SA_PATH)
required_cols = {"bus_id", "charge_capacity_MW", "storage_capacity_mwh"}
missing = required_cols - set(df_base.columns)
if missing:
    raise ValueError(f"storage_assignment.csv missing columns: {missing}")

total_charge = float(df_base["charge_capacity_MW"].sum())
total_energy = float(df_base["storage_capacity_mwh"].sum())
print(f"\nBaseline totals -> charge={total_charge:.3f} MW, energy={total_energy:.3f} MWh")

# 2) Load bus_prop (Zone labels)
if BUS_PROP_PATH is None:
    candidates = [
        f"{PROJECT_DIR}/data/grid/bus_prop_boyuan.csv",       # has BUS_ZONE letters
        f"{PROJECT_DIR}/data/grid/bus_prop_liu_etal_2024.csv" # may not have BUS_ZONE
    ]
    BUS_PROP_PATH = next((p for p in candidates if os.path.exists(p)), None)

if BUS_PROP_PATH is None or not os.path.exists(BUS_PROP_PATH):
    raise FileNotFoundError(
        "Could not find bus_prop CSV. Set BUS_PROP_PATH explicitly, or ensure one of:\n  "
        + f"{PROJECT_DIR}/data/grid/bus_prop_boyuan.csv\n  "
        + f"{PROJECT_DIR}/data/grid/bus_prop_liu_etal_2024.csv\nexists."
    )

print("BUS_PROP_PATH:", BUS_PROP_PATH)
df_bus = pd.read_csv(BUS_PROP_PATH)

# Prefer BUS_ZONE if present (lettered zones A/B/…); fall back to ZONE if it contains letters.
zone_col = None
if "BUS_ZONE" in df_bus.columns:
    zone_col = "BUS_ZONE"
elif "ZONE" in df_bus.columns and df_bus["ZONE"].astype(str).str.isalpha().any():
    zone_col = "ZONE"

if zone_col is None:
    raise ValueError(
        "No lettered zone column found. Expected BUS_ZONE (A, B, …). "
        "Use bus_prop_boyuan.csv or provide a mapping."
    )

if "BUS_I" not in df_bus.columns:
    raise ValueError("bus_prop file missing BUS_I column.")

zone_a_buses = (
    df_bus.loc[df_bus[zone_col].astype(str).str.upper().str.strip() == "A", "BUS_I"]
    .dropna()
    .astype(int)
    .tolist()
)
if not zone_a_buses:
    raise ValueError("No Zone A buses found. Check zone labels or bus_prop file.")

print(f"Zone A bus count: {len(zone_a_buses)}")
print("First few Zone A buses:", zone_a_buses[:10])

# 3) Randomly split ALL capacity across Zone A buses (reproducible)
rng = np.random.default_rng(42)
w = rng.random(len(zone_a_buses))
w = w / w.sum()

out = pd.DataFrame({"bus_id": zone_a_buses})
out["charge_capacity_MW"]   = (total_charge * w).round(6)
out["storage_capacity_mwh"] = (total_energy * w).round(6)

# exact-sum correction to remove rounding drift
out.at[out["charge_capacity_MW"].idxmax(), "charge_capacity_MW"] += (total_charge - out["charge_capacity_MW"].sum())
out.at[out["storage_capacity_mwh"].idxmax(), "storage_capacity_mwh"] += (total_energy - out["storage_capacity_mwh"].sum())

# 4) Backup old file and overwrite
ts = time.strftime("%Y%m%d-%H%M%S")
backup = f"{INPUTS}/storage_assignment.backup-{ts}.csv"
df_base.to_csv(backup, index=False)
out_path = SA_PATH
out.sort_values("bus_id").to_csv(out_path, index=False)

print("\n=== DONE ===")
print("Backup written to:", backup)
print("New storage_assignment.csv written to:", out_path)
print(
    f"Check totals -> charge={out['charge_capacity_MW'].sum():.6f} (was {total_charge:.6f}), "
    f"energy={out['storage_capacity_mwh'].sum():.6f} (was {total_energy:.6f})"
)

=== PATHS IN USE ===
PROJECT_DIR: /home/fs01/jl2966/acorn-julia2/acorn-julia
RUN_NAME   : low_RE_mod_elec_iter0
RUN_DIR    : /home/fs01/jl2966/acorn-julia2/acorn-julia/runs/low_RE_mod_elec_iter0
INPUTS     : /home/fs01/jl2966/acorn-julia2/acorn-julia/runs/low_RE_mod_elec_iter0/inputs
SA_PATH    : /home/fs01/jl2966/acorn-julia2/acorn-julia/runs/low_RE_mod_elec_iter0/inputs/storage_assignment.csv

Baseline totals -> charge=5999.894 MW, energy=13360.258 MWh
BUS_PROP_PATH: /home/fs01/jl2966/acorn-julia2/acorn-julia/data/grid/bus_prop_boyuan.csv
Zone A bus count: 8
First few Zone A buses: [54, 55, 56, 57, 58, 59, 60, 61]

=== DONE ===
Backup written to: /home/fs01/jl2966/acorn-julia2/acorn-julia/runs/low_RE_mod_elec_iter0/inputs/storage_assignment.backup-20251028-001658.csv
New storage_assignment.csv written to: /home/fs01/jl2966/acorn-julia2/acorn-julia/runs/low_RE_mod_elec_iter0/inputs/storage_assignment.csv
Check totals -> charge=5999.893968 (was 5999.893968), energy=13360.258408 (was 13

In [38]:
PROJ = Path("/home/fs01/jl2966/acorn-julia2/acorn-julia")
RUN  = PROJ / "runs" / "low_RE_mod_elec_iter0"     # change per scenario
BUSPROP_CSV = PROJ / "data" / "grid" / "bus_prop_boyuan.csv"
BASE_STORAGE = RUN / "inputs" / "storage_assignment.csv"
OUT_STORAGE  = RUN / "inputs" / "storage_assignment_seasonal.csv"

ZONE_FILTER = "A"           # which BUS_ZONE to target
WEIGHT_BY_PD = False        # True = PD-weighted split, False = uniform

# seasonal tech defaults
SEASONAL_EFF = 0.65
SEASONAL_INIT_SOC = 0.50
SEASONAL_END_SOC_MIN = 0.30
SEASONAL_DISCH_COST = 0.0   # set >0.0 to penalize throughput if wanted

bus = pd.read_csv(BUSPROP_CSV)
zone_buses = bus.loc[bus["BUS_ZONE"].astype(str).str.upper()==ZONE_FILTER, "BUS_I"].astype(int)

base = pd.read_csv(BASE_STORAGE)
total_MW  = base["charge_capacity_MW"].sum()
total_MWh = base["storage_capacity_mwh"].sum()

# weights across chosen zone buses
if WEIGHT_BY_PD and "PD" in bus.columns:
    pd_map = bus.set_index("BUS_I")["PD"].reindex(zone_buses).fillna(0).clip(lower=0)
    if pd_map.sum() == 0:
        w = pd.Series(1.0, index=zone_buses) / len(zone_buses)
    else:
        w = pd_map / pd_map.sum()
else:
    w = pd.Series(1.0, index=zone_buses) / len(zone_buses)

out = pd.DataFrame({
    "bus_id": zone_buses.values,
    "charge_capacity_MW": total_MW * w.values,
    "storage_capacity_mwh": total_MWh * w.values,
    "tech": "SEASONAL",
    "eff": SEASONAL_EFF,
    "init_soc": SEASONAL_INIT_SOC,
    "end_soc_min": SEASONAL_END_SOC_MIN,
    "discharge_cost_per_mwh": SEASONAL_DISCH_COST,
})

out.to_csv(OUT_STORAGE, index=False)
print(f"Wrote {OUT_STORAGE} with {len(out)} rows (Zone {ZONE_FILTER})")


Wrote /home/fs01/jl2966/acorn-julia2/acorn-julia/runs/low_RE_mod_elec_iter0/inputs/storage_assignment_seasonal.csv with 8 rows (Zone A)


## Storage

In [ ]:
# Generate random sites per zone
df_genX_battery = pu.map_genX_zones_to_nyiso(df_genX[df_genX['Resource'] == 'battery'].copy(), genX_zone_col='Zone')
df_genX_battery = pi.generate_random_sites(df_genX_battery,
                                           sites_per_zone = 3,
                                           columns_to_scale = ['EndEnergyCap', 'EndCap'])

In [ ]:
# Assign to random bus
df_genX_battery = pu.nearest_neighbor_lat_lon(df_genX_battery, PV_bus_only=False)

In [ ]:
# Store
(
    df_genX_battery[['bus_id', 'EndCap', 'EndEnergyCap']]
    .groupby('bus_id').sum()
    .rename(columns={'EndCap':'charge_capacity_MW',
                     'EndEnergyCap':'storage_capacity_mwh'})
    .to_csv(f"{run_path}/inputs/storage_assignment.csv")
)

## Thermal

In [ ]:
# Match thermal capacities (natural gas only)
df_ng_matched = pi.match_ng_capacity(df_genX,
                                     df_genprop,
                                     store_path = run_path,
                                     retirement_method='highest_cost_first')

In [ ]:
# Store with only online generators
df_ng_matched = df_ng_matched[df_ng_matched['GEN_STATUS'] == 1].copy()
df_ng_matched.to_csv(f"{run_path}/inputs/genprop_ng_matched.csv", index=False)

## Nuclear

In [ ]:
# GenX
df_genX.query('Resource == "nuclear"')[['Resource', 'Zone', 'EndCap']]

In [ ]:
# ACORN
df_genprop[df_genprop['FUEL_TYPE'] == 'UR'][['GEN_NAME', 'PMAX', 'GEN_ZONE', 'GEN_STATUS']]

In [ ]:
# Match nuclear by hand -> keep only nuclear in zone C
# Retire all nuclear outside zone C
df_genprop_nuclear = df_genprop[(df_genprop['FUEL_TYPE'] == 'UR') & (df_genprop['GEN_ZONE'] == 'C')]
# Store
df_genprop_nuclear.to_csv(f"{run_path}/inputs/genprop_nuclear_matched.csv", index=False)

## Hydro

In [ ]:
# First store the hydro generators
large_hydro_names = ['Moses Niagara (Fleet)', 'St Lawrence - FDR (Fleet)']

df_genprop_hydro = df_genprop[df_genprop['GEN_NAME'].isin(large_hydro_names)]
df_genprop_hydro.to_csv(f"{run_path}/inputs/genprop_hydro.csv", index=False)

In [ ]:
# Get hourly hydro to buses
hydro_scenario = climate_scenario_years.split('_')[0]
df_small_hydro, df_large_hydro = hu.assign_hydro_GD_to_buses(hydro_scenario = hydro_scenario)

In [ ]:
# Get max cap
acorn_small_hydro_cap = float(df_small_hydro.groupby('datetime').sum().max())
acorn_large_hydro_cap = float(df_large_hydro.groupby('datetime').sum().max() / (7*24))

In [ ]:
# Store
df_small_hydro.reset_index().pivot(index='bus_id', 
                                   columns='datetime', 
                                   values='power_MW').to_csv(f"{run_path}/inputs/small_hydro_{hydro_scenario}.csv")

df_large_hydro.reset_index().pivot(index='bus_id', 
                                   columns='datetime', 
                                   values='power_predicted_mwh').to_csv(f"{run_path}/inputs/large_hydro_{hydro_scenario}.csv")

## Nameplate capacity comparison

In [ ]:
# Construct plot df
genX_resources = df_genX['Resource'].unique()
genX_capacity = df_genX.groupby('Resource')['EndCap'].sum()

acorn_capacity = {'battery': genX_capacity['battery'],
                  'biomass': 0.0,
                  'hydroelectric': acorn_small_hydro_cap + acorn_large_hydro_cap,
                  'distributed_generation': genX_capacity['distributed_generation'],
                  'hydroelectric_storage': 0.0,
                  'natural_gas': df_ng_matched['PMAX'].sum(), 
                  'nuclear': df_genprop_nuclear['PMAX'].sum(),
                  'offshore_wind': genX_capacity['offshore_wind'],
                  'onshore_wind_existing': genX_capacity['onshore_wind_existing'],
                  'water_heating': 0.0,
                  'solar_existing': genX_capacity['solar_existing'],
                  'transportation': 0.0,
                  'solar_new': genX_capacity['solar_new'], 
                 }

df_cap_plot = pd.DataFrame({'GenX': [genX_capacity[res] for res in genX_resources],
                            'ACORN': [acorn_capacity[res] for res in genX_resources]},
                           index=genX_resources)

In [ ]:
fig, ax = plt.subplots(figsize=(10,4))

df_cap_plot.plot.bar(rot=-45, ax=ax)

ax.grid(alpha=0.5)
ax.set_ylabel('Nameplate capacity [MW]')

ax.set_title(f'ACORN total: {df_cap_plot['ACORN'].sum():.0f} MW, GenX total: {df_cap_plot['GenX'].sum():.0f} MW')

plt.savefig(f"{run_path}/figs/capacity_comparison.pdf", bbox_inches='tight')

# Renewable resources

## Solar

### DPV

In [ ]:
#######################
# Existing solar: DPV
#######################
# Generate existing sites
df_genX_solar_dpv = pu.map_genX_zones_to_nyiso(df_genX[df_genX['Resource'] == 'distributed_generation'].copy(),
                                               genX_zone_col='Zone')

df_genX_solar_dpv = pi.generate_random_sites(df_genX_solar_dpv, sites_per_zone=10)

# Correction factors
correction_file = f"{project_path}/data/solar/models/tgw_solar_correction_factors.csv"

# Calculate the timeseries
df_solar_dpv = su.calculate_solar_timeseries_from_genX(
    df_genX = df_genX_solar_dpv,
    climate_paths = climate_paths,
    correction_file = correction_file,
    PV_bus_only = False
)

# Store
df_solar_dpv.reset_index().pivot(index='bus_id', 
                             columns='datetime', 
                             values='power_MW').to_csv(f"{run_path}/inputs/solar_dpv_{climate_scenario_years}.csv")

In [ ]:
# Plot DPV
fig, axs = plt.subplots(1, 2, figsize=(12,5), layout='constrained')
fig.suptitle('Solar DPV capacity by zone')

# Map
ax=axs[0]
nyiso_gdf.plot(ax=ax, column='zone', cmap='Paired')
sns.scatterplot(data=df_genX_solar_dpv, x='longitude', y='latitude', size='EndCap', color='black', ax=ax)
leg = ax.legend(title='Capacity [MW]')

# Bar
ax=axs[1]
df_genX_solar_dpv.groupby('zone')['EndCap'].sum().plot.bar(ax=axs[1], rot=0)
ax.set_ylabel('Capacity [MW]')
ax.grid(alpha=0.2)

plt.savefig(f"{run_path}/figs/solar_dpv.pdf", bbox_inches='tight')

### UPV

In [ ]:
######################
# Existing solar: UPV
######################
# Generate existing sites
df_genX_solar_existing = pu.map_genX_zones_to_nyiso(df_genX[df_genX['Resource'] == 'solar_existing'].copy(),
                                                   genX_zone_col='Zone')

df_genX_solar_existing = pi.generate_random_sites(df_genX_solar_existing, sites_per_zone=1)

# Correction factors
correction_file = f"{project_path}/data/solar/models/tgw_solar_correction_factors.csv"

# Calculate the timeseries
df_solar_existing = su.calculate_solar_timeseries_from_genX(
    df_genX = df_genX_solar_existing,
    climate_paths = climate_paths,
    correction_file = correction_file,
)

In [ ]:
# ################################################################
# # NOTE: The genX zones do not always align with their coordinates
# # (i.e. they will be assigned to a different zone).
# # When aggregating to bus level, assigning `match_zone` = True (default)
# # means that genX zones will be respected. `match_zone` = False
# # means it will use the lat/lon coords instead.
# ################################################################
# # Read NYISO shapefile
# nyiso_gdf = gpd.read_file(f'{project_path}/data/nyiso/gis/NYISO_Load_Zone_Dissolved.shp')

# # Plot
# fig, axs = plt.subplots(1,2, figsize=(15,10))

# nyiso_gdf.plot(ax=axs[0])
# pu.merge_to_zones(df_genX_ds[df_genX_ds['Resource_Type'] == 'Utility_PV'],
#                lat_name='latitude',
#                lon_name='longitude').plot(ax=axs[0], column='region', legend=True, cmap='Paired')

# nyiso_gdf.plot(ax=axs[1], column='zone', legend=True, cmap='Paired')

# plt.show()

In [ ]:
####################
# New solar: UPV
####################
# Update the genX regions
df_genX_ds_solar = df_genX_ds[df_genX_ds['technology'] == 'UtilityPV_Class1_Moderate_'].copy()
df_genX_ds_solar = pu.map_genX_zones_to_nyiso(df_genX_ds_solar, C_and_E_mapping='C')

# Correction factors
correction_file = f"{project_path}/data/solar/models/tgw_solar_correction_factors.csv"

# Calculate the timeseries
df_solar_new = su.calculate_solar_timeseries_from_genX(
    df_genX = df_genX_ds_solar,
    climate_paths = climate_paths,
    correction_file = correction_file,
)

In [ ]:
# Combined solar
df_solar_total = pd.concat([df_solar_existing, df_solar_new])
df_solar_total = df_solar_total.reset_index().groupby(['bus_id', 'datetime']).sum()

# Store combined
df_solar_total.reset_index().pivot(index='bus_id', 
                             columns='datetime', 
                             values='power_MW').to_csv(f"{run_path}/inputs/solar_upv_{climate_scenario_years}.csv")

In [ ]:
# Plot UPV
fig, axs = plt.subplots(1, 2, figsize=(12,5), layout='constrained')
fig.suptitle('Solar UPV capacity by zone')

df_solar_upv_plot = pd.concat([df_genX_ds_solar, df_genX_solar_existing])

# Map
ax = axs[0]

nyiso_gdf.plot(ax=ax, column='zone', cmap='Paired')
sns.scatterplot(data=df_solar_upv_plot, x='longitude', y='latitude', size='EndCap', color='black', ax=ax)
leg = ax.legend(title='Capacity [MW]')

# Bar
ax = axs[1]
df_solar_upv_plot.groupby('genX_zone')['EndCap'].sum().plot.bar(ax=axs[1], rot=0)
ax.set_ylabel('Capacity [MW]')
ax.grid(alpha=0.2)

plt.savefig(f"{run_path}/figs/solar_upv.pdf", bbox_inches='tight')

## Wind

### Onshore

In [ ]:
########################
# Existing onshore wind
########################
# Generate existing onshore sites
df_genX_onshore_existing = pu.map_genX_zones_to_nyiso(df_genX[df_genX['Resource'] == 'onshore_wind_existing'].copy(),
                                                      genX_zone_col='Zone')

df_genX_onshore_existing = wu.generate_onshore_wind_sites(df_genX_onshore_existing)

# Stability coefficients
stab_coef_file = f"{pu.project_path}/data/wind/models/tgw_stability_coefficients_2007-2013_every5.csv"

# Calculate the timeseries
df_onshore_wind = wu.calculate_wind_timeseries_from_genX(
    df_genX = df_genX_onshore_existing,
    climate_paths = climate_paths,
    stab_coef_file = stab_coef_file,
    iec_curve = 'iec1',
)

In [ ]:
# Plot onshore wind
fig, axs = plt.subplots(1, 2, figsize=(12,5), layout='constrained')
fig.suptitle('Onshore wind capacity by zone')

# Map
ax = axs[0]

nyiso_gdf.plot(ax=ax, column='zone', cmap='Paired')
sns.scatterplot(data=df_genX_onshore_existing.round(2), x='longitude', y='latitude', size='EndCap', color='black', ax=ax)
leg = ax.legend(title='Capacity [MW]', loc='upper left')

# Bar
ax = axs[1]
df_genX_onshore_existing.groupby('zone')['EndCap'].sum().plot.bar(ax=axs[1], rot=0)
ax.set_ylabel('Capacity [MW]')
ax.grid(alpha=0.2)

plt.savefig(f"{run_path}/figs/onshore_wind.pdf", bbox_inches='tight')

### Offshore

In [ ]:
### Generate sites
# Generate offshore locations
df_genX_offshore_existing = pu.map_genX_zones_to_nyiso(df_genX[df_genX['Resource'] == 'offshore_wind'].copy(),
                                                      genX_zone_col='Zone')

df_genX_offshore_existing = wu.generate_offshore_wind_sites(df_genX_offshore_existing)

# Plot offshore wind
fig, axs = plt.subplots(1, 2, figsize=(12,5), layout='constrained')
fig.suptitle('Offshore wind capacity by zone')

# Map
ax = axs[0]

nyiso_gdf[nyiso_gdf['zone'].isin(['J', 'K'])].plot(ax=ax, column='zone', cmap='Paired')
sns.scatterplot(data=df_genX_offshore_existing.round(2), x='longitude', y='latitude', size='EndCap', color='black', ax=ax)
leg = ax.legend(title='Capacity [MW]', loc='upper left')

# Bar
ax = axs[1]
df_genX_offshore_existing.groupby('Zone')['EndCap'].sum().plot.bar(ax=axs[1], rot=0)
ax.set_ylabel('Capacity [MW]')
ax.grid(alpha=0.2)

plt.savefig(f"{run_path}/figs/offshore_wind.pdf", bbox_inches='tight')

In [ ]:
## Generate timeseries
# Stability coefficients
stab_coef_file = f"{pu.project_path}/data/wind/models/tgw_stability_coefficients_2007-2013_every5.csv"

# Calculate the timeseries
df_offshore_wind = wu.calculate_wind_timeseries_from_genX(
    df_genX = df_genX_offshore_existing,
    climate_paths = climate_paths,
    stab_coef_file = stab_coef_file,
    iec_curve = 'offshore',
)

In [ ]:
# Combined wind
df_wind_total = pd.concat([df_onshore_wind, df_offshore_wind])
df_wind_total = df_wind_total.reset_index().groupby(['bus_id', 'datetime']).sum()

# Store combined
df_wind_total.reset_index().pivot(index='bus_id', 
                             columns='datetime', 
                             values='power_MW').to_csv(f"{run_path}/inputs/wind_{climate_scenario_years}.csv")